**Table of contents**<a id='toc0_'></a>    
1. [Data Sources](#toc1_)    
2. [Linking datasets](#toc2_)    
2.1. [LSOA -> Ward](#toc2_1_)    
2.2. [Household composition](#toc2_2_)    
2.2.1. [Converting raw data to long format, standarise cateogries, map LSOA to Ward, and merging them](#toc2_2_1_)    
2.2.2. [Create ward-level data](#toc2_2_2_)    
2.3. [Household size](#toc2_3_)    
2.3.1. [Converting raw data to long format, map LSOA to Ward, and merging them](#toc2_3_1_)    
2.4. [Create ward-level data](#toc2_4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path('./data')

# 1. <a id='toc1_'></a>[Data Sources](#toc0_)

Data can be downloaded from the [Nomis website](https://www.nomisweb.co.uk/). Below are links to the relevant datasets for each census year.

2001:
<!-- Univariate tables](https://www.nomisweb.co.uk/sources/census_2001_uv)
[Standard Tables](https://www.nomisweb.co.uk/sources/census_2001_st)
[Census 2001 Tables](https://www.nomisweb.co.uk/census/2001/data_finder)
[UV065 Household composition (households)](https://www.nomisweb.co.uk/census/2001/uv065)
[ST051 Tenure and household size by number of rooms](https://www.nomisweb.co.uk/census/2001/st051) -->
[UV046 - Household composition](https://www.nomisweb.co.uk/datasets/uv046)

[UV051 - Number of people living in households](https://www.nomisweb.co.uk/datasets/uv051)

2011:

[QS406EW Household size](https://www.nomisweb.co.uk/census/2011/qs406ew)

[KS105EW Household composition](https://www.nomisweb.co.uk/census/2011/ks105ew)

2021:

[TS017 - Household size](https://www.nomisweb.co.uk/datasets/c2021ts017)

[TS003 - Household composition](https://www.nomisweb.co.uk/datasets/c2021ts003)

Key parameters:
- Geography: Super Output Areas - Lower Layer, Bradford
- Household Composition / Size: select all available categories (all levels)
- Measure: download counts (numbers) rather than percentages, as percentages will be calculated later

To get ward-level data, we also need following lookup tables:

The 2001 LSOA to 2011 LSOA lookup table can be downloaded from [ONS website](https://geoportal.statistics.gov.uk/search?q=Lower%20Layer%20Super%20Output%20Area%20(2001)%20to%20Lower%20Layer%20Super%20Output%20Area%20(2011)%20to%20Local%20Authority%20District%20(2011)%20Lookup)

The 2011 LSOA to 2021 LSOA lookup table can be downloaded from [ONS website](https://geoportal.statistics.gov.uk/datasets/ons::lsoa-2011-to-lsoa-2021-to-local-authority-district-2022-exact-fit-lookup-for-ew-v3/about)

The 2021 LSOA to 2024 Ward lookup table can be downloaded from [ONS website](https://geoportal.statistics.gov.uk/datasets/ons::lsoa-2021-to-electoral-ward-2024-to-lad-2024-best-fit-lookup-in-ew/about)

# 2. <a id='toc2_'></a>[Linking datasets](#toc0_)

## 2.1. <a id='toc2_1_'></a>[LSOA -> Ward](#toc0_)

In [2]:
lookup_tables = {
    '01OA-11OA': {
        'filename': 'Lower_Layer_Super_Output_Area_(2001)_to_Lower_Layer_Super_Output_Area_(2011)_to_Local_Authority_District_(2011)_Lookup_in_England_and_Wales.csv',
        'columns': ['LSOA01CD', 'LSOA11CD'],
    },
    '11OA-21OA': {
        'filename': 'LSOA_(2011)_to_LSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V3).csv',
        'columns': ['LSOA11CD', 'LSOA21CD'],
    },
    '21OA-24Ward': {
        'filename': 'LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.csv',
        'columns': ['LSOA21CD', 'WD24NM'],
    }
}

dfs_lookup = {
    name: pd.read_csv(
        data_dir / info['filename'],
        usecols=info['columns']
    ).drop_duplicates()
    for name, info in lookup_tables.items()
}

In [3]:
# create lsoa - 2021 ward lookup table 
ward_mapping = (
    dfs_lookup['01OA-11OA'][['LSOA01CD', 'LSOA11CD']]
    .merge(dfs_lookup['11OA-21OA'], on='LSOA11CD', how='left')
    .merge(dfs_lookup['21OA-24Ward'], on='LSOA21CD', how='left')
)

ward_mapping = ward_mapping.drop_duplicates()

In [4]:
def lsoa_to_ward(df, year, lsoa_col, ward_col, ward_mapping):
    df = df.copy()
    ward_mapping = ward_mapping.copy()
    year = str(year)

    lsoa_cd_col = {
        "2001": "LSOA01CD",
        "2011": "LSOA11CD",
        "2021": "LSOA21CD"
    }

    source_lsoa_col = lsoa_cd_col[year]

    # clean LSOA codes
    df[lsoa_col] = df[lsoa_col].astype(str).str.strip()
    ward_mapping[source_lsoa_col] = ward_mapping[source_lsoa_col].astype(str).str.strip()

    lookup_table = (
        ward_mapping
        .loc[
            ward_mapping[source_lsoa_col].isin(df[lsoa_col]),
            [source_lsoa_col, "WD24NM"]
        ]
        .drop_duplicates()
    )

    # identify one-to-many LSOA -> ward cases
    one_to_many = (
        lookup_table
        .loc[lookup_table.duplicated(subset=source_lsoa_col, keep=False)]
        .sort_values(source_lsoa_col)
    )

    if not one_to_many.empty:
        print(
            f"Warning: {one_to_many[source_lsoa_col].nunique()} {source_lsoa_col} values "
            f"map to multiple WD24NM values. These will be split equally using weights."
        )
        display(one_to_many)

    # add equal-split weight
    lookup_table["n_wards"] = (
        lookup_table
        .groupby(source_lsoa_col)["WD24NM"]
        .transform("nunique")
    )

    lookup_table["weight"] = 1 / lookup_table["n_wards"]

    # merge instead of map, because one LSOA can map to multiple wards
    df_out = df.merge(
        lookup_table[[source_lsoa_col, "WD24NM", "weight"]],
        left_on=lsoa_col,
        right_on=source_lsoa_col,
        how="left"
    )

    # rename ward column if needed
    df_out = df_out.rename(columns={"WD24NM": ward_col})

    # drop duplicated source LSOA column if it is different from df lsoa_col
    if source_lsoa_col != lsoa_col and source_lsoa_col in df_out.columns:
        df_out = df_out.drop(columns=[source_lsoa_col])

    return df_out

## 2.2. <a id='toc2_2_'></a>[Household composition](#toc0_)

### 2.2.1. <a id='toc2_2_1_'></a>[Converting raw data to long format, standarise cateogries, map LSOA to Ward, and merging them](#toc0_)

First, we load source data and xlsx file for harmonising categories.


::: {.callout-note title="Harmonisation note"}

By default, the harmonised categories use the 2021 Census household composition categories as the reference classification. Categories from 2001 and 2011 are mapped onto the closest equivalent 2021 category wherever possible.

Categories are merged when earlier censuses provide a more detailed breakdown than 2021. For example, where 2001 distinguishes between households with one dependent child and households with two or more dependent children, these categories are combined into the broader 2021 category "with dependent children". Similarly, older "other household" subcategories, such as all-student, all-pensioner, and other households, are merged where 2021 reports them as a combined category.

A harmonised category is renamed or differs from the exact 2021 wording where the 2021 label does not fully represent the earlier census definitions, or where a broader comparable category is needed across all years. This mainly applies to categories affected by definitional changes, such as "pensioner", "aged 65 and over", and "aged 66 years and over", or categories where civil partnerships were introduced into later census classifications.

:::

In [5]:
# load source data
data_mapping = {
    '2001': {
        'filename': 'UV046 Household composition 2001.xlsx',
        'skiprows': 7,
    },
    '2011': {
        'filename': 'KS105EW Household composition 2011.xlsx',
        'skiprows': 8,
    },
    '2021': {
        'filename': 'TS003 Household Composition 2021.xlsx',
        'skiprows': 7,
    }
}

dfs = {
    year: pd.read_excel(data_dir / info['filename'], skiprows=info['skiprows'])
    for year, info in data_mapping.items()
}

# load category mappings
category_mapping = pd.read_excel(data_dir / "household_composition_category_mapping.xlsx").dropna(how='all')
category_mapping = category_mapping.rename(columns=str)

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


The cateogires will be mapped in to harmonised one based on following table:

In [6]:
category_mapping

,Harmonised category,2001,2011,2021
0,Total: All households,All categories: Household composition,All categories: Household composition,Total: All households
2,One‑person household,One person,One person household,One-person household
3,One-person household: Older person,One person: Pensioner,One person household: Aged 65 and over,One-person household: Aged 66 years and over
4,One‑person household: Other,One person: Other,One person household: Other,One-person household: Other
6,Single family household,One family and no others,One family household,Single family household
7,Single-family household: All older persons,One family and no others: All pensioner,One family only: All aged 65 and over,Single family household: All aged 66 years and...
8,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
9,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
10,Single family household: Married or civil part...,One family and no others: Married Couple House...,One family only: Married or same-sex civil par...,Single family household: Married or civil part...
11,Single family household: Married or civil part...,One family and no others: Married Couple House...,NaN,NaN


In [7]:
dfs_long = []

for year, df in dfs.items():
    print(year)
    df_long = []
    lsoa_col = "lsoa"
    ward_col = "ward"
    num_col = "number"
    pct_col = "pct"
    raw_cat_col = "raw_category"
    curated_cat_col = "curated_category"
    main_cat_col = "main_category"
    sub_cat_col = "sub_category"
    total_col = "Total: All households"
    
    # extract lsoa
    raw_oa_col = [col for col in df.columns if 'lower layer' in col]
    df[lsoa_col] = df[raw_oa_col[0]].str.split(':').str[0].str.strip()
 
    df_long = df.drop(columns=raw_oa_col).melt(id_vars=lsoa_col, var_name=raw_cat_col, value_name=num_col).dropna(subset=[num_col])
    
    # map lsoa to 2021 ward
    df_long = lsoa_to_ward(df_long, year, lsoa_col, ward_col, ward_mapping)
    
    # curate categories
    cat_mapping = category_mapping[[year, 'Harmonised category']].dropna().set_index(year)['Harmonised category']
    df_long[curated_cat_col] = df_long[raw_cat_col].str.strip().map(cat_mapping).fillna(df_long[raw_cat_col].str.strip())
    
    # change the position of columns
    front_cols = [lsoa_col, ward_col]
    df_long = df_long[front_cols + [c for c in df_long.columns if c not in front_cols]]
       
    is_total = df_long[curated_cat_col].isin([total_col])
    parts = df_long.loc[~is_total, curated_cat_col].str.split(":")
    num_categories = parts.str.len().max()
    df_long[main_cat_col] = df_long[curated_cat_col]
    df_long.loc[~is_total, main_cat_col] = parts.str[0].str.strip()
    for i in range(1, num_categories):
        df_long[f"{sub_cat_col}{i}"] = parts.str[i].str.strip()    
    
    totals = df.rename(columns={'All categories: Household composition': total_col})[[lsoa_col, total_col]].rename(columns={total_col: "total_households"})
    df_long = df_long.merge(totals, on=lsoa_col, how="left")
    df_long[pct_col] = (df_long[num_col] / df_long["total_households"] * 100).round(1)
    # df_long = df_long.drop(columns=["total_households"])
    
    df_long['census'] = year
    
    end_cols = [num_col, "weight", "census"]
    
    df_long = df_long[[c for c in df_long.columns if c not in end_cols] + end_cols]
    
    dfs_long.append(df_long)
    

df_comp_lsoa = pd.concat(dfs_long, ignore_index=True)
df_comp_lsoa.to_excel(data_dir / 'household_composition_lsoa.xlsx', index=False)

2001


,LSOA01CD,WD24NM
3584,E01010823,City
3588,E01010823,Manningham


2011
2021


In [8]:
df_comp_lsoa

,lsoa,ward,raw_category,curated_category,main_category,sub_category1,sub_category2,total_households,pct,number,weight,census
0,E01010646,Craven,All categories: Household composition,Total: All households,Total: All households,NaN,NaN,1157.0,100.0,1157.0,1.0,2001
1,E01010647,Craven,All categories: Household composition,Total: All households,Total: All households,NaN,NaN,1073.0,100.0,1073.0,1.0,2001
2,E01010648,Craven,All categories: Household composition,Total: All households,Total: All households,NaN,NaN,1370.0,100.0,1370.0,1.0,2001
3,E01010692,Ilkley,All categories: Household composition,Total: All households,Total: All households,NaN,NaN,1492.0,100.0,1492.0,1.0,2001
4,E01010691,Ilkley,All categories: Household composition,Total: All households,Total: All households,NaN,NaN,1443.0,100.0,1443.0,1.0,2001
...,...,...,...,...,...,...,...,...,...,...,...,...
21687,E01033692,City,"Other household types: Other, including all fu...","Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,612.0,3.6,22.0,1.0,2021
21688,E01010606,Bowling and Barkerend,"Other household types: Other, including all fu...","Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,591.0,5.2,31.0,1.0,2021
21689,E01010842,Manningham,"Other household types: Other, including all fu...","Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,630.0,3.7,23.0,1.0,2021
21690,E01033691,City,"Other household types: Other, including all fu...","Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,1008.0,4.7,47.0,1.0,2021


### 2.2.2. <a id='toc2_2_2_'></a>[Create ward-level data](#toc0_)

::: {.callout-note title="Weighted allocation"}

Some LSOAs are matched to more than one 2024 ward because of boundary changes. To avoid double counting, the LSOA-level count for each household composition category is not copied in full to every matched ward. Instead, each category-specific count is multiplied by an allocation weight.

Where no population- or household-based weights are available, an equal-split weight is used:

`weight = 1 / number of matched wards for the LSOA`

The allocated value is then calculated as:

`weighted_value = category_count_in_LSOA × weight`

For example, if one LSOA maps to two wards, each ward receives 50% of the original LSOA count for each household composition category. Ward-level category counts are then calculated by summing these weighted values across all matched LSOAs.

:::

In [9]:
tmp = df_comp_lsoa.drop(columns=[lsoa_col, pct_col])

tmp[f"weighted_{num_col}"] = tmp[num_col] * tmp["weight"]
tmp['weighted_total_households'] = tmp['total_households'] * tmp["weight"]

unique_keys = [ward_col, curated_cat_col, 'census']

df_ward = (
    tmp
    .groupby(unique_keys, as_index=False)
    .agg(
        **{
            num_col: (f"weighted_{num_col}", "sum"),
            "total_households": ('weighted_total_households', "sum")
        }
    )
)

df_ward[pct_col] = (df_ward[num_col] / df_ward["total_households"] * 100).round(1)
# df_comp_ward = df_comp_ward.drop(columns=["total_households"])

df_comp_ward = df_comp_lsoa.drop(columns=[lsoa_col, pct_col, 'weight', 'total_households', num_col, pct_col, raw_cat_col]).drop_duplicates()

assert df_comp_ward[df_comp_ward.duplicated(subset=unique_keys, keep=False)].empty

df_comp_ward = df_comp_ward.merge(df_ward, how='left', on=unique_keys)

front_cols = [ward_col, curated_cat_col]
end_cols = [num_col, pct_col, "census"]
other_cols = [c for c in df_comp_ward.columns if c not in front_cols + end_cols]

df_comp_ward = df_comp_ward[front_cols + other_cols + end_cols]

df_comp_ward.to_excel(data_dir / 'household_composition_ward.xlsx', index=False)

In [10]:
df_comp_ward

,ward,curated_category,main_category,sub_category1,sub_category2,total_households,number,pct,census
0,Craven,Total: All households,Total: All households,NaN,NaN,15556.0,15556.0,100.0,2001
1,Ilkley,Total: All households,Total: All households,NaN,NaN,13338.0,13338.0,100.0,2001
2,Wharfedale,Total: All households,Total: All households,NaN,NaN,11039.0,11039.0,100.0,2001
3,Keighley East,Total: All households,Total: All households,NaN,NaN,14849.0,14849.0,100.0,2001
4,Keighley Central,Total: All households,Total: All households,NaN,NaN,15304.0,15304.0,100.0,2001
...,...,...,...,...,...,...,...,...,...
1855,Little Horton,"Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,6740.0,231.0,3.4,2021
1856,Queensbury,"Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,6408.0,123.0,1.9,2021
1857,Wibsey,"Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,6489.0,153.0,2.4,2021
1858,Royds,"Other household types: Other, including all fu...",Other household types,"Other, including all full-time students and al...",NaN,7276.0,165.0,2.3,2021


## 2.3. <a id='toc2_3_'></a>[Household size](#toc0_)

In [11]:
data_mapping = {
    '2001': {
        'filename': 'UV051  Household size 2001.xlsx',
        'skiprows': 7,
    },
    '2011': {
        'filename': 'QS406EW Household size 2011.xlsx',
        'skiprows': 8,
    },
    '2021': {
        'filename': 'TS017 Household  Size 2021.xlsx',
        'skiprows': 7,
    }
}

dfs = {
    year: pd.read_excel(data_dir / info['filename'], skiprows=info['skiprows'])
    for year, info in data_mapping.items()
}

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### 2.3.1. <a id='toc2_3_1_'></a>[Converting raw data to long format, map LSOA to Ward, and merging them](#toc0_)

In [12]:
dfs_long = []

for year, df in dfs.items():
    print(year)
    df_long = []
    lsoa_col = 'lsoa'
    ward_col = "ward"
    num_col = "number"
    pct_col = "pct"
    main_cat_col = "main_category"
    total_col = "Total: All households"
    
    df.rename(columns={"All categories: Household size" : total_col}, inplace=True)
    
    # extract lsoa
    raw_oa_col = [col for col in df.columns if 'lower layer' in col]
    df[lsoa_col] = df[raw_oa_col[0]].str.split(':').str[0].str.strip()
 
    df_long = df.drop(columns=raw_oa_col).melt(id_vars=lsoa_col, var_name=main_cat_col, value_name=num_col).dropna(subset=[num_col])
    
    # map lsoa to 2021 ward
    df_long = lsoa_to_ward(df_long, year, lsoa_col, ward_col, ward_mapping)
    
    totals = df[[lsoa_col, total_col]].rename(columns={total_col: "total_households"})
    df_long = df_long.merge(totals, on=lsoa_col, how="left")
    df_long[pct_col] = (df_long[num_col] / df_long["total_households"] * 100).round(1)
    # df_long = df_long.drop(columns=["total_households"])
    
    df_long['census'] = year
    
    front_cols = [lsoa_col, ward_col, main_cat_col]
    end_cols = [num_col, "weight", "census"]
    other_cols = [c for c in df_long.columns if c not in front_cols + end_cols]
    
    df_long = df_long[front_cols + other_cols + end_cols]
    
    dfs_long.append(df_long)
    

df_size_lsoa = pd.concat(dfs_long, ignore_index=True)

df_size_lsoa = df_size_lsoa[~(df_size_lsoa[main_cat_col] == '0 people in household')] # they all zeros

df_size_lsoa.to_excel(data_dir / 'household_size_lsoa.xlsx', index=False)

2001


,LSOA01CD,WD24NM
3584,E01010823,City
3588,E01010823,Manningham


2011
2021


## 2.4. <a id='toc2_4_'></a>[Create ward-level data](#toc0_)

In [13]:
tmp = df_size_lsoa.drop(columns=[lsoa_col, pct_col])

tmp[f"weighted_{num_col}"] = tmp[num_col] * tmp["weight"]
tmp['weighted_total_households'] = tmp['total_households'] * tmp["weight"]

unique_keys = [ward_col, main_cat_col, 'census']

df_ward = (
    tmp
    .groupby(unique_keys, as_index=False)
    .agg(
        **{
            num_col: (f"weighted_{num_col}", "sum"),
            "total_households": ('weighted_total_households', "sum")
        }
    )
)

df_ward[pct_col] = (df_ward[num_col] / df_ward["total_households"] * 100).round(1)
# df_comp_ward = df_comp_ward.drop(columns=["total_households"])

df_size_ward = df_size_lsoa.drop(columns=[lsoa_col, pct_col, 'weight', 'total_households', num_col, pct_col]).drop_duplicates()

assert df_size_ward[df_size_ward.duplicated(subset=unique_keys, keep=False)].empty

df_size_ward = df_size_ward.merge(df_ward, how='left', on=unique_keys)

front_cols = [ward_col, main_cat_col]
end_cols = [num_col, pct_col, "census"]
other_cols = [c for c in df_size_ward.columns if c not in front_cols + end_cols]

df_size_ward = df_size_ward[front_cols + other_cols + end_cols]

df_size_ward.to_excel(data_dir / 'household_size_ward.xlsx', index=False)

df_size_ward

,ward,main_category,total_households,number,pct,census
0,Craven,Total: All households,6754.0,6754.0,100.0,2001
1,Ilkley,Total: All households,5867.0,5867.0,100.0,2001
2,Wharfedale,Total: All households,4554.0,4554.0,100.0,2001
3,Keighley East,Total: All households,6085.0,6085.0,100.0,2001
4,Keighley Central,Total: All households,5508.0,5508.0,100.0,2001
...,...,...,...,...,...,...
805,Little Horton,8 or more people in household,6745.0,188.0,2.8,2021
806,Queensbury,8 or more people in household,6419.0,18.0,0.3,2021
807,Wibsey,8 or more people in household,6476.0,36.0,0.6,2021
808,Royds,8 or more people in household,7279.0,37.0,0.5,2021
